# Specific heat animation (single patient)

Animate specific heat curves across time windows (dev-scale).

**Legacy notebooks merged:**
- specific_heat_animation.ipynb

In [ ]:
%matplotlib inline
from lrgsglib.config.funcs import move_to_rootf
move_to_rootf(pathname="lrg_eegfc")
from lrg_eegfc.notebook import *

In [ ]:
from lrgsglib.core import entropy, get_giant_component_leftoff
from lrgsglib.utils.basic.signals import bandpass_sos
import networkx as nx

patient = list_patients(Path('data/stereoeeg_patients'))[0]
phase = PHASE_LABELS[0]
band = BRAIN_BANDS_NAMES[0]

recording = load_patient_dataset_robust(patient, Path('data/stereoeeg_patients'), phases=[phase])[phase]
timeseries = recording.timeseries
fs = float(recording.parameters.get('fs'))

window_sec = 10.0
overlap = 0.25
window_len = int(window_sec * fs)
step = int(window_len * (1.0 - overlap))

n_samples = timeseries.shape[1]
indices = list(range(0, n_samples - window_len + 1, step))
indices = indices[:12]

low, high = BRAIN_BANDS[band]

entropy_curves = []
for start in indices:
    window = timeseries[:, start:start + window_len]
    filtered = bandpass_sos(window, low, high, fs, 4)
    corr = build_corr_network(filtered, filter_type='abs', zero_diagonal=True)
    corr[corr < 0.9] = 0
    G = nx.from_numpy_array(corr)
    Gcc, _ = get_giant_component_leftoff(G)
    _, C, _, tau = entropy(Gcc)
    entropy_curves.append((tau[1:], C))

len(entropy_curves)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Build common tau grid and interpolate
all_tau = np.unique(np.concatenate([t for t, _ in entropy_curves]))
log_tau = np.log(all_tau)

interp_curves = []
for t, C in entropy_curves:
    interp_curves.append(np.interp(log_tau, np.log(t), C))

frames = []
for idx in range(len(interp_curves) - 1):
    a = interp_curves[idx]
    b = interp_curves[idx + 1]
    for alpha in np.linspace(0, 1, 10, endpoint=False):
        frames.append((1 - alpha) * a + alpha * b)
frames.append(interp_curves[-1])

fig, ax = plt.subplots()
line, = ax.plot([], [], lw=2)
ax.set_xscale('log')
ax.set_xlim(all_tau.min(), all_tau.max())
ax.set_ylim(np.min(interp_curves), np.max(interp_curves))
ax.set_xlabel('tau')
ax.set_ylabel('C (specific heat)')


def update(frame_data):
    line.set_data(all_tau, frame_data)
    return (line,)

ani = FuncAnimation(fig, update, frames=frames, interval=100, blit=True)
ani